In [ ]:
from time import sleep
import requests
import pandas as pd
from os import path  # Note: Your teacher uses path.join() below, which requires this or 'import os'


In [ ]:
base_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query"

In [ ]:
def make_request(base_url, params):
    response = requests.get(base_url, params = params)
    if response:    #True if the HTTP status code is between 200 and 299                
        return response.json()
    else:
        raise Exception("Error Downloading JSON")

In [ ]:
# The active API endpoint verified in class
base_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query"

#params = {'limit': 0}
#n_stations = make_request(base_url, params)['total_count']
#print(n_stations)


# Define the query parameters for your active MapServer URL
params = {
    "where": "1=1",   # is always true
    "outFields": "*", # * is a wildcard that literally means everythin, All the Fields 
    "f": "json"
}

# 1. Use the teacher's function to download the active dictionary structure
api_response = make_request(base_url, params=params)

# 2. Get the total count by measuring the length of the 'features' list
n_stations = len(api_response['features'])   # Contains tall the row per station 

# 3. Print the result (This will output 11, matching your teacher's console!)
#print(n_stations)
print(f"Total Number of Active Stations = {n_stations}")




In [ ]:

max_records_per_request = 100

current_time = 0
sleep_time = 60 * 60   #sleep_time = 60 * 60: This equals 3600 seconds (exactly 1 hour). The script will sleep for an hour between data collections.
total_time = (24 * 60 * 60) + sleep_time #total_time = (24 * 60 * 60) + sleep_time: 
                                         #This sets up a 25-hour total window 86400 seconds + 3600  { seconds). 
                                         #This ensures the script collects data across a full 24-hour cycle before closing automatically.


path_csv = ['res', 'valencia_pollution_dataset.csv'] 
path_csv_solved = path.join(*path_csv)   # path.join(*path_csv): This is a clean, cross-platform way to build file directories. 
                                         # The asterisk * unpacks the list, turning it into path.join('res', 'valencia_pollution_dataset.csv').
                                         # On Windows, this creates a path using backslashes (res\valencia_pollution_dataset.csv),
                                         # while on Mac/Linux it uses forward slashes (res/valencia_pollution_dataset.csv), 
                                         # preventing path syntax errors!


In [44]:
# We can simulate the loop structure. For a single test run, let's look at the data collection inside:
while current_time < total_time:
    # 1. Initialize an empty list for this hour's poll
    pollution_list = []
    
    # 2. Define active parameters for our server
    params = {
        "where": "1=1",
        "outFields": "objectid,nombre,direccion,tipozona,tipoemisio,no2,pm10",
        "f": "json"
    }
    
    # 3. Call the teacher's function to get the live data dictionary
    api_response = make_request(base_url, params=params)
    
    # 4. Extract the list of records from 'features'
    stations_data = api_response.get('features', [])
    
    # 5. Loop through them to clean up the nesting (removing the 'attributes' wrapper)
    for station in stations_data:
        # Appending just the clean dictionary inside 'attributes' to match the teacher's console output
        pollution_list.append(station['attributes'])
        
    # 6. Print the clean results list to the console
    print(f"--- Data collected at simulated timestamp: {current_time} seconds ---")
    print(pollution_list[:2])  # Display just the first 2 items to keep it clean
    
    # --- Control mechanism to prevent infinite loops in your notebook ---
    # For now, we increment current_time by the total time to simulate a single clean execution.
    current_time += total_time

--- Data collected at simulated timestamp: 0 seconds ---
[{'objectid': 20, 'nombre': 'Cabanyal', 'direccion': 'CABANYAL', 'tipozona': 'Urbana', 'tipoemisio': 'Fondo', 'no2': 14.0, 'pm10': 34.0}, {'objectid': 21, 'nombre': 'Olivereta', 'direccion': 'OLIVERETA', 'tipozona': 'Urbana', 'tipoemisio': 'Tráfico', 'no2': 37.0, 'pm10': 50.0}]


In [ ]:
# Teacher 
first_time = True  
current_time = 0
while current_time < total_time:
    offset = 0
    pollution_list = []

    while offset < n_stations:
        params = {"limit": max_records_per_request,
                  "offset": offset}

        sub_list = make_request(base_url, params)['features']
        pollution_list += sub_list

        offset += max_records_per_request

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")

    df = pd.DataFrame(pollution_list,
                      columns=['objectid', 'nombre', 'direccion', 'tipozona',
                               'tipoemision', 'so2', 'no2', 'o3', 'co',
                               'pm10', 'pm25', 'fecha_carga', 'cal'])
    display(df)
    df.to_csv(path_csv_solved, sep=';', header=True if first_time else False, index=False, mode='a')

    first_time = False
    sleep(sleep_time)


In [ ]:
# CRITICAL FIX: Initialize this variable before starting the loop!
first_time = True  
current_time = 0

while current_time < total_time:
    offset = 0
    pollution_list = []

    # Using the n_stations we calculated earlier (11)
    while offset < n_stations:
        # Define query parameters for your active MapServer
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "json"
        }

        # FIX 1: Change ['results'] to ['features'] to match the new API structure
        sub_list = make_request(base_url, params)['features']
        
        # FIX 2: Extract the inner ['attributes'] dictionary from each feature item
        # to match the clean database row format your teacher's DataFrame expects.
        clean_rows = [station['attributes'] for station in sub_list]
        pollution_list += clean_rows

        # We increase the offset manually to break the inner loop safely
        offset += max_records_per_request

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")

    # Construct the DataFrame using the matching columns from your new endpoint
    df = pd.DataFrame(pollution_list,
                      columns=['objectid', 'nombre', 'direccion', 'tipozona',
                               'tipoemisio', 'so2', 'no2', 'o3', 'co',
                               'pm10', 'pm25', 'fecha_carg', 'calidad_am'])
    display(df)
    
    # Save the dataframe to your CSV path
    df.to_csv(path_csv_solved, sep=';', header=True if first_time else False, index=False, mode='a')

    # Update loop states
    first_time = False
    
    # FIX 3: Increment current_time so the loop can eventually reach total_time and stop!
    current_time += sleep_time
    
    # Pause execution for 1 hour before making the next API pull
    sleep(sleep_time)

Current Time = 0, Records Processed = 11


,objectid,nombre,direccion,tipozona,tipoemisio,so2,no2,o3,co,pm10,pm25,fecha_carg,calidad_am
0,20,Cabanyal,CABANYAL,Urbana,Fondo,NaN,14.0,NaN,NaN,34.0,12.0,1780088400000,Razonablemente Buena
1,21,Olivereta,OLIVERETA,Urbana,Tráfico,NaN,37.0,NaN,NaN,50.0,17.0,1780088400000,Regular
2,14,Boulevar Sur,BULEVARD SUD,Urbana,Tráfico,1.0,22.0,94.0,NaN,NaN,NaN,1780088400000,Razonablemente Buena
3,15,Molí del Sol,MOLÍ DEL SOL,Suburbana,Tráfico,3.0,12.0,98.0,0.0,38.0,10.0,1780088400000,Razonablemente Buena
4,17,Universidad Politécnica,POLITÈCNIC,Suburbana,Fondo,1.0,13.0,90.0,NaN,19.0,9.0,1780088400000,Razonablemente Buena
5,18,Viveros,VIVERS,Urbana,Fondo,3.0,14.0,113.0,NaN,NaN,NaN,1780088400000,Regular
6,16,Pista de Silla,PISTA DE SILLA,Urbana,Tráfico,7.0,12.0,61.0,0.0,32.0,11.0,1780088400000,Razonablemente Buena
7,19,Centro,VALÈNCIA CENTRE,Urbana,Tráfico,NaN,16.0,NaN,NaN,38.0,12.0,1780088400000,Razonablemente Buena
8,12,Dr. Lluch,DR.LLUCH,Urbana,Tráfico,NaN,19.0,NaN,NaN,40.0,13.0,1780088400000,Razonablemente Buena
9,22,Patraix,PATRAIX,Urbana,Tráfico,NaN,53.0,NaN,NaN,38.0,14.0,1780088400000,Razonablemente Buena


In [ ]:

print("------------.")
current_time = 0
while current_time < total_time:
    offset = 0
    pollution_list = []

    while offset < n_stations:
        params = {"limit": max_records_per_request,
                  "offset": offset}

        sub_list = make_request(base_url, params)['results']
        pollution_list += sub_list

        offset += max_records_per_request

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")

    df = pd.DataFrame(pollution_list,
print("------------.")